# Notebook 10 — S2 Controlled Perturbation Robustness

This notebook evaluates the **frozen S1 pipelines** under controlled TEST-input perturbations.

- **S2-M:** 10% MCAR missingness.
- **S2-N:** Gaussian noise with `sigma_noise = 0.05 × TRAIN scale` for each variable.
- **S2-C:** combined missingness + noise.
- Targets are never perturbed.
- Missing values are causally forward-filled inside the 5-step history; leading gaps use TRAIN means.
- Perturbation seeds: `42, 123, 456, 789, 2026`.
- Frozen deep-model training seeds: `42, 123, 2026`.
- No S2 hyperparameter tuning.
- Perturbations are generated once on the common unscaled `5 × 7 × 3` physical-history tensor, then mapped to sequential and tabular representations.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT=Path.cwd().resolve()
O=ROOT/"outputs"
D07=O/"07D_final_alignment"; D07C=O/"07C_graph"
D08A=O/"08A_tabular_models"; D08B=O/"08B_lstm_tcn"; D08C=O/"08C_gcn_graph_ablation"
OUT=O/"10_S2_robustness"; OUT.mkdir(parents=True,exist_ok=True)
for p in [D07,D07C,D08A,D08B,D08C]: assert p.exists(), f"Missing: {p}"

PSEEDS=[42,123,456,789,2026]; TSEEDS=[42,123,2026]
MISSING_RATE=.10; NOISE_FRACTION=.05
NODES=7; STEPS=5; CHANNELS=3
VARIABLES=["temperature","relative_humidity","co2"]
NTEST=13790; NSNAP=1970
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:",ROOT); print("DEVICE:",device); print("OUT:",OUT)

## 1. Load and audit the frozen 07D representations

In [ ]:
Xg_raw=np.load(D07/"X_graph_test_unscaled.npy").astype(np.float32)
yg=np.load(D07/"y_graph_test.npy").astype(np.float32)
mg=pd.read_csv(D07/"metadata_graph_test.csv")
Xs_orig=np.load(D07/"X_sequence_test_unscaled.npy").astype(np.float32)
ys=np.load(D07/"y_sequence_test.npy").astype(np.float32)
ms=pd.read_csv(D07/"metadata_sequence_test.csv")
assert Xg_raw.shape==(NSNAP,STEPS,NODES,CHANNELS) and yg.shape==(NSNAP,NODES)
assert Xs_orig.shape==(NTEST,STEPS,CHANNELS) and ys.shape==(NTEST,)

nodes=pd.read_csv(D07C/"01_graph_node_index.csv").sort_values("node_index")
uuid2node=dict(zip(nodes.uuid,nodes.node_index))
mg["target_timestamp"]=pd.to_datetime(mg["target_timestamp"],utc=True)
ms["target_timestamp"]=pd.to_datetime(ms["target_timestamp"],utc=True)
snap={ts:i for i,ts in enumerate(mg.target_timestamp)}
seqmap=np.array([(snap[r.target_timestamp],uuid2node[r.uuid]) for r in ms.itertuples()],dtype=np.int64)

Xs_check=np.stack([Xg_raw[s,:,n,:] for s,n in seqmap])
print("max alignment diff:",np.max(np.abs(Xs_check-Xs_orig)))
assert np.allclose(Xs_check,Xs_orig,rtol=0,atol=1e-5)

seqsc=pd.read_csv(D07/"06_sequence_scaler.csv").set_index("feature")
grsc=pd.read_csv(D07/"07_graph_scaler.csv").set_index("feature")
tabsc=pd.read_csv(D07/"04_tabular_core_scaler.csv").set_index("feature")
seq_mean=seqsc.loc[VARIABLES,"train_mean"].to_numpy(np.float32)
seq_scale=seqsc.loc[VARIABLES,"train_scale"].to_numpy(np.float32)
gr_mean=grsc.loc[VARIABLES,"train_mean"].to_numpy(np.float32)
gr_scale=grsc.loc[VARIABLES,"train_scale"].to_numpy(np.float32)
noise_sigma=.05*gr_scale
print("noise sigma T/RH/CO2:",noise_sigma)
print("✓ common TEST alignment verified")

## 2. Reconstruct frozen S1 tabular models from TRAIN only

In [ ]:
TARGET="target_temperature_t_plus_15"
META=["uuid","timestamp","target_timestamp","split","segment_id",TARGET]
du=pd.read_csv(D07/"02_tabular_core_15_unscaled.csv")
ds=pd.read_csv(D07/"03_tabular_core_15_scaled.csv")
features=[c for c in du.columns if c not in META]
assert len(features)==15
tru=du[du.split=="train"]; trs=ds[ds.split=="train"]
Xtru=tru[features].to_numpy(np.float32); Xtrs=trs[features].to_numpy(np.float32)
ytr=tru[TARGET].to_numpy(np.float32)

ridge=Ridge(alpha=.01).fit(Xtrs,ytr)
rf=RandomForestRegressor(n_estimators=300,max_depth=20,min_samples_leaf=4,
                         max_features=.8,random_state=42,n_jobs=-1).fit(Xtru,ytr)
xgbr=xgb.XGBRegressor(n_estimators=500,max_depth=3,learning_rate=.03,
                      subsample=.9,colsample_bytree=.9,objective="reg:squarederror",
                      random_state=42,n_jobs=-1,tree_method="hist",
                      reg_lambda=1.0,verbosity=0).fit(Xtru,ytr)
tab_mean=tabsc.loc[features,"train_mean"].to_numpy(np.float32)
tab_scale=tabsc.loc[features,"train_scale"].to_numpy(np.float32)
print("✓ tabular S1 configurations reconstructed; no hyperparameter search")

## 3. Frozen deep-learning architectures and checkpoints

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self,hidden=64,layers=1,dropout=0.0):
        super().__init__(); self.lstm=nn.LSTM(3,hidden,num_layers=layers,batch_first=True,
        dropout=dropout if layers>1 else 0.0); self.head=nn.Linear(hidden,1)
    def forward(self,x):
        z,_=self.lstm(x); return self.head(z[:,-1,:]).squeeze(-1)

class CausalConv1d(nn.Module):
    def __init__(self,cin,cout,kernel=2,dilation=1):
        super().__init__(); self.pad=(kernel-1)*dilation; self.conv=nn.Conv1d(cin,cout,kernel,dilation=dilation)
    def forward(self,x): return self.conv(nn.functional.pad(x,(self.pad,0)))

class TCNRegressor(nn.Module):
    def __init__(self,channels=64,kernel=2,dropout=.1):
        super().__init__(); self.net=nn.Sequential(
        CausalConv1d(3,channels,kernel,1),nn.ReLU(),nn.Dropout(dropout),
        CausalConv1d(channels,channels,kernel,2),nn.ReLU(),nn.Dropout(dropout))
        self.head=nn.Linear(channels,1)
    def forward(self,x):
        z=self.net(x.transpose(1,2)); return self.head(z[:,:,-1]).squeeze(-1)

class GraphConv(nn.Module):
    def __init__(self,fin,fout): super().__init__(); self.lin=nn.Linear(fin,fout)
    def forward(self,x,A): return self.lin(torch.einsum("ij,bjf->bif",A,x))

class TemporalGCN(nn.Module):
    def __init__(self,hidden=64,dropout=.1):
        super().__init__(); self.gc1=GraphConv(15,hidden); self.gc2=GraphConv(hidden,hidden)
        self.drop=nn.Dropout(dropout); self.head=nn.Linear(hidden,1)
    def forward(self,x,A):
        x=x.permute(0,2,1,3).contiguous().view(x.size(0),7,15)
        z=self.drop(torch.relu(self.gc1(x,A))); z=self.drop(torch.relu(self.gc2(z,A)))
        return self.head(z).squeeze(-1)

def load_seq(fam,seed):
    m=LSTMRegressor(64,1,0.0) if fam=="LSTM" else TCNRegressor(64,2,.1)
    p=D08B/f"{fam}_seed{seed}.pt"; assert p.exists(),p
    m.load_state_dict(torch.load(p,map_location=device,weights_only=True)); return m.to(device).eval()

def canon(e):
    e=np.asarray(e)
    if e.size==0:return np.empty((2,0),dtype=np.int64)
    return e.astype(np.int64) if e.shape[0]==2 else e.T.astype(np.int64)

def ahat(e):
    A=np.zeros((7,7),np.float32)
    for a,b in e.T:
        if a!=b:A[a,b]=A[b,a]=1
    A+=np.eye(7,dtype=np.float32); d=A.sum(1); q=d**-.5
    return torch.tensor(q[:,None]*A*q[None,:],dtype=torch.float32,device=device)

edgefiles={"G_native":"edge_index_native.npy","G_operational":"edge_index_operational.npy",
           "G_shuffled":"edge_index_shuffled_degree_preserving.npy"}
Ah={k:ahat(canon(np.load(D07/v))) for k,v in edgefiles.items()}

def load_gcn(topo,seed):
    m=TemporalGCN(64,.1); p=D08C/f"{topo}_GCN_h64_seed{seed}.pt"; assert p.exists(),p
    m.load_state_dict(torch.load(p,map_location=device,weights_only=True)); return m.to(device).eval()
print("✓ checkpoint paths verified")

## 4. Perturbation, causal imputation, representation mapping and scaling

In [ ]:
def causal_impute(X):
    Y=X.copy()
    for t in range(STEPS):
        miss=np.isnan(Y[:,t,:,:])
        if t>0:Y[:,t,:,:]=np.where(miss,Y[:,t-1,:,:],Y[:,t,:,:])
    for c in range(3):
        a=Y[:,:,:,c]; a[np.isnan(a)]=gr_mean[c]; Y[:,:,:,c]=a
    assert not np.isnan(Y).any()
    return Y.astype(np.float32)

def perturb(condition,seed):
    rng=np.random.default_rng(seed); X=Xg_raw.copy(); mask=np.zeros(X.shape,bool)
    if condition in ["S2-M","S2-C"]:
        mask=rng.random(X.shape)<MISSING_RATE; X[mask]=np.nan
    if condition in ["S2-N","S2-C"]:
        noise=rng.normal(0,noise_sigma.reshape(1,1,1,3),X.shape).astype(np.float32)
        X=np.where(np.isnan(X),np.nan,X+noise)
    return causal_impute(X),{"condition":condition,"perturbation_seed":seed,
        "masked_values":int(mask.sum()),"total_values":int(mask.size),
        "realized_missing_pct":100*float(mask.mean())}

def to_seq(X): return np.stack([X[s,:,n,:] for s,n in seqmap]).astype(np.float32)
def to_tab_frozen_order(Xseq):
    """
    Convert a perturbed sequential representation into the
    exact frozen 07D tabular TEST row order.
    """
    Xseq_aligned = Xseq[seq_rows_for_tab]

    return Xseq_aligned.reshape(
        len(Xseq_aligned),
        15
    ).astype(np.float32)
def sc_seq(X): return ((X-seq_mean.reshape(1,1,3))/seq_scale.reshape(1,1,3)).astype(np.float32)
def sc_graph(X): return ((X-gr_mean.reshape(1,1,1,3))/gr_scale.reshape(1,1,1,3)).astype(np.float32)
def sc_tab(X): return ((X-tab_mean)/tab_scale).astype(np.float32)
print("✓ perturbation protocol ready")

## 5. Inference helpers and S1 reproduction audit

In [ ]:
# ============================================================
# CELL 5 — INFERENCE HELPERS AND FINAL S1 REPRODUCTION AUDIT
# ============================================================

def met(y, p):
    y = np.asarray(y).reshape(-1)
    p = np.asarray(p).reshape(-1)

    e = p - y

    return {
        "n": len(y),
        "MAE": float(mean_absolute_error(y, p)),
        "RMSE": float(np.sqrt(mean_squared_error(y, p))),
        "R2": float(r2_score(y, p)),
        "Bias": float(e.mean()),
        "Error_SD": float(e.std(ddof=1))
    }


def pred_seq(model, X, batch_size=1024):
    out = []

    model.eval()

    with torch.no_grad():
        for i in range(0, len(X), batch_size):

            xb = torch.from_numpy(
                X[i:i + batch_size]
            ).to(device)

            pred = model(xb)

            out.append(
                pred.cpu().numpy()
            )

    return np.concatenate(out)


def pred_graph(model, X, A, batch_size=256):
    out = []

    model.eval()

    with torch.no_grad():
        for i in range(0, len(X), batch_size):

            xb = torch.from_numpy(
                X[i:i + batch_size]
            ).to(device)

            pred = model(xb, A)

            out.append(
                pred.cpu().numpy()
            )

    return np.concatenate(out, axis=0)


# ============================================================
# 1. TARGET
# ============================================================

y = ys.astype(np.float32)

assert len(y) == NTEST

print("Target observations:", len(y))


# ============================================================
# 2. GRAPH -> SEQUENTIAL
# ============================================================

Xs0 = to_seq(Xg_raw)

assert Xs0.shape == (NTEST, 5, 3)

max_seq_diff = float(
    np.max(
        np.abs(
            Xs0 - Xs_orig
        )
    )
)

print(
    "Max sequence alignment difference:",
    max_seq_diff
)

assert np.allclose(
    Xs0,
    Xs_orig,
    rtol=0,
    atol=1e-5
)

print(
    "✓ Exact Graph -> Sequential alignment passed"
)


# ============================================================
# 3. CREATE EXACT TABULAR/SEQUENTIAL KEY MAPPING
# ============================================================

seq_keys = ms[
    ["uuid", "target_timestamp"]
].copy()

seq_keys["target_timestamp"] = pd.to_datetime(
    seq_keys["target_timestamp"],
    utc=True
)

seq_keys["seq_row"] = np.arange(
    len(seq_keys)
)


tab_test = du.loc[
    du["split"] == "test"
].copy()

tab_test["target_timestamp"] = pd.to_datetime(
    tab_test["target_timestamp"],
    utc=True
)

tab_test["tab_row"] = np.arange(
    len(tab_test)
)


# ------------------------------------------------------------
# Uniqueness audit
# ------------------------------------------------------------

assert not seq_keys.duplicated(
    ["uuid", "target_timestamp"]
).any()

assert not tab_test.duplicated(
    ["uuid", "target_timestamp"]
).any()


# ------------------------------------------------------------
# Exact one-to-one matching
# ------------------------------------------------------------

mapping = tab_test[
    [
        "uuid",
        "target_timestamp",
        "tab_row"
    ]
].merge(
    seq_keys[
        [
            "uuid",
            "target_timestamp",
            "seq_row"
        ]
    ],
    on=[
        "uuid",
        "target_timestamp"
    ],
    how="left",
    validate="one_to_one"
)


print(
    "\nTabular TEST rows :",
    len(tab_test)
)

print(
    "Sequence TEST rows:",
    len(seq_keys)
)

print(
    "Matched rows      :",
    mapping["seq_row"].notna().sum()
)


assert len(mapping) == NTEST

assert mapping[
    "seq_row"
].notna().all()


# ============================================================
# 4. REORDER SEQUENCE ROWS TO EXACT FROZEN TABULAR ORDER
# ============================================================

seq_rows_for_tab = (
    mapping["seq_row"]
    .astype(int)
    .to_numpy()
)


Xs0_tab_order = Xs0[
    seq_rows_for_tab
]


Xt0 = (
    Xs0_tab_order
    .reshape(
        len(Xs0_tab_order),
        15
    )
    .astype(np.float32)
)


Xt_frozen = (
    tab_test[features]
    .to_numpy(np.float32)
)


# ============================================================
# 5. STRICT TABULAR VALUE AUDIT
# ============================================================

difference = np.abs(
    Xt0 - Xt_frozen
)


print(
    "\nMaximum absolute difference:",
    float(difference.max())
)

print(
    "Mean absolute difference:",
    float(difference.mean())
)

print(
    "Number of unequal values:",
    int(
        np.sum(
            difference > 1e-5
        )
    )
)


assert np.allclose(
    Xt0,
    Xt_frozen,
    rtol=0,
    atol=1e-5
)


print(
    "\n✓ Exact alignment by "
    "(uuid, target_timestamp) passed"
)

print(
    "✓ Common aligned TEST samples:",
    len(Xt0)
)


# ============================================================
# 6. SCALE EACH REPRESENTATION USING FROZEN TRAIN PARAMETERS
# ============================================================

# Sequential keeps its native sequence order
Xss0 = sc_seq(
    Xs0
)

# Tabular uses the exact frozen tabular row order
Xts0 = sc_tab(
    Xt0
)

# Graph keeps its native snapshot/node organization
Xgs0 = sc_graph(
    Xg_raw
)


# ============================================================
# 7. TABULAR TARGET IN TABULAR ROW ORDER
# ============================================================

y_tab = (
    tab_test[TARGET]
    .to_numpy(np.float32)
)

assert len(y_tab) == NTEST


# ============================================================
# 8. VERIFY TARGET EQUIVALENCE AFTER KEY ALIGNMENT
# ============================================================

# ys is in sequence order.
# Reorder it to frozen tabular order.
y_seq_to_tab = y[
    seq_rows_for_tab
]

target_diff = float(
    np.max(
        np.abs(
            y_seq_to_tab - y_tab
        )
    )
)

print(
    "\nMaximum target difference after alignment:",
    target_diff
)

assert np.allclose(
    y_seq_to_tab,
    y_tab,
    rtol=0,
    atol=1e-6
)

print(
    "✓ Targets are identical after key-based alignment"
)


# ============================================================
# 9. PERSISTENCE COLUMN
# ============================================================

idx_temperature_t = features.index(
    "temperature_t"
)

print(
    "\nPersistence column:",
    idx_temperature_t,
    features[idx_temperature_t]
)


# ============================================================
# 10. DETERMINISTIC S1 MODELS
# ============================================================

S1 = {}


S1["Persistence"] = Xt0[
    :,
    idx_temperature_t
]


S1["Ridge"] = ridge.predict(
    Xts0
)


S1["RandomForest"] = rf.predict(
    Xt0
)


S1["XGBoost"] = xgbr.predict(
    Xt0
)


# ============================================================
# 11. LSTM AND TCN
# ============================================================

for fam in [
    "LSTM",
    "TCN"
]:

    predictions = []

    for seed in TSEEDS:

        model = load_seq(
            fam,
            seed
        )

        p = pred_seq(
            model,
            Xss0
        )

        assert len(p) == NTEST

        predictions.append(
            p.astype(np.float32)
        )

    S1[fam] = np.stack(
        predictions,
        axis=0
    )


# ============================================================
# 12. GCN TOPOLOGIES
# ============================================================

for topo in [
    "G_native",
    "G_operational",
    "G_shuffled"
]:

    predictions = []

    for seed in TSEEDS:

        model = load_gcn(
            topo,
            seed
        )

        pm = pred_graph(
            model,
            Xgs0,
            Ah[topo]
        )

        assert pm.shape == (
            NSNAP,
            NODES
        )

        # Convert graph snapshot/node ordering
        # to sequence node-level ordering.
        p_seq_order = np.array(
            [
                pm[s, n]
                for s, n in seqmap
            ],
            dtype=np.float32
        )

        assert len(
            p_seq_order
        ) == NTEST

        predictions.append(
            p_seq_order
        )

    S1[topo] = np.stack(
        predictions,
        axis=0
    )


# ============================================================
# 13. FROZEN S1 DETERMINISTIC REPRODUCTION
# ============================================================

expected = {
    "Persistence": 0.111809,
    "Ridge": 0.113392,
    "RandomForest": 0.203773,
    "XGBoost": 0.241443
}


print(
    "\n"
    + "=" * 70
)

print(
    "FROZEN S1 DETERMINISTIC REPRODUCTION"
)

print(
    "=" * 70
)


for model_name, expected_rmse in expected.items():

    result = met(
        y_tab,
        S1[model_name]
    )

    got = result[
        "RMSE"
    ]

    difference_rmse = abs(
        got - expected_rmse
    )

    print(
        f"{model_name:15s} "
        f"RMSE={got:.6f} | "
        f"expected={expected_rmse:.6f} | "
        f"diff={difference_rmse:.8f}"
    )

    assert difference_rmse < 5e-4, (
        model_name,
        got,
        expected_rmse
    )


print(
    "\n✓ S1 deterministic reproduction passed"
)


# ============================================================
# 14. STOCHASTIC MODEL AUDIT
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "FROZEN S1 STOCHASTIC MODEL AUDIT"
)

print(
    "=" * 70
)


# Sequential predictions are currently in sequence order.
# Their metrics must therefore use y.
for fam in [
    "LSTM",
    "TCN"
]:

    rmse_values = []

    for i, seed in enumerate(
        TSEEDS
    ):

        result = met(
            y,
            S1[fam][i]
        )

        rmse_values.append(
            result["RMSE"]
        )

        print(
            f"{fam:15s} "
            f"seed={seed:<4d} "
            f"RMSE={result['RMSE']:.6f}"
        )

    print(
        f"{fam:15s} "
        f"mean RMSE="
        f"{np.mean(rmse_values):.6f} "
        f"± "
        f"{np.std(rmse_values, ddof=1):.6f}"
    )


# Graph predictions have been converted to sequence order.
for topo in [
    "G_native",
    "G_operational",
    "G_shuffled"
]:

    rmse_values = []

    for i, seed in enumerate(
        TSEEDS
    ):

        result = met(
            y,
            S1[topo][i]
        )

        rmse_values.append(
            result["RMSE"]
        )

        print(
            f"{topo:15s} "
            f"seed={seed:<4d} "
            f"RMSE={result['RMSE']:.6f}"
        )

    print(
        f"{topo:15s} "
        f"mean RMSE="
        f"{np.mean(rmse_values):.6f} "
        f"± "
        f"{np.std(rmse_values, ddof=1):.6f}"
    )


print(
    "\n"
    + "=" * 70
)

print(
    "✓ ALL S1 ALIGNMENT AND REPRODUCTION AUDITS COMPLETED"
)

print(
    "✓ Common TEST population:",
    NTEST
)

print(
    "✓ No S2 perturbation has been applied yet"
)

print(
    "=" * 70
)

In [ ]:
# ============================================================
# CELL 6 — EXECUTE S2-M, S2-N AND S2-C
# ============================================================

rows = []
audits = []

conditions = [
    "S2-M",   # 10% MCAR missingness
    "S2-N",   # Gaussian noise
    "S2-C"    # Missingness + Gaussian noise
]

print("=" * 70)
print("S2 CONTROLLED PERTURBATION ROBUSTNESS")
print("=" * 70)

print("Perturbation seeds :", PSEEDS)
print("Training seeds     :", TSEEDS)
print("Missing rate       :", MISSING_RATE)
print("Noise fraction     :", NOISE_FRACTION)
print("TEST observations  :", NTEST)

print()


# ============================================================
# LOOP OVER S2 CONDITIONS
# ============================================================

for cond in conditions:

    print("\n" + "=" * 70)
    print("CONDITION:", cond)
    print("=" * 70)

    for psd in PSEEDS:

        print(
            f"\nRunning {cond} | "
            f"perturbation seed = {psd}"
        )

        # ----------------------------------------------------
        # 1. APPLY PERTURBATION TO COMMON GRAPH TENSOR
        # ----------------------------------------------------

        Xgr, audit_info = perturb(
            cond,
            psd
        )

        audits.append(
            audit_info
        )


        # ----------------------------------------------------
        # 2. GRAPH -> SEQUENTIAL REPRESENTATION
        # ----------------------------------------------------

        Xsr = to_seq(
            Xgr
        )

        assert Xsr.shape == (
            NTEST,
            STEPS,
            CHANNELS
        )


        # ----------------------------------------------------
        # 3. SEQUENTIAL -> EXACT FROZEN TABULAR ORDER
        # ----------------------------------------------------

        Xtr = to_tab_frozen_order(
            Xsr
        )

        assert Xtr.shape == (
            NTEST,
            15
        )


        # ----------------------------------------------------
        # 4. SCALE USING TRAIN-ONLY FROZEN PARAMETERS
        # ----------------------------------------------------

        Xss = sc_seq(
            Xsr
        )

        Xts = sc_tab(
            Xtr
        )

        Xgs = sc_graph(
            Xgr
        )


        # ====================================================
        # 5. DETERMINISTIC TABULAR MODELS
        # ====================================================

        deterministic_predictions = {
            "Persistence":
                Xtr[:, idx_temperature_t],

            "Ridge":
                ridge.predict(Xts),

            "RandomForest":
                rf.predict(Xtr),

            "XGBoost":
                xgbr.predict(Xtr)
        }


        # IMPORTANT:
        # Tabular predictions are in frozen TABULAR row order.
        # Therefore metrics must use y_tab.

        for model_name, prediction in (
            deterministic_predictions.items()
        ):

            result = met(
                y_tab,
                prediction
            )

            rows.append({
                "condition": cond,
                "perturbation_seed": psd,
                "training_seed": np.nan,
                "model": model_name,
                **result
            })


        # ====================================================
        # 6. LSTM AND TCN
        # ====================================================

        # Sequential predictions remain in SEQUENCE row order.
        # Therefore metrics use y.

        for fam in [
            "LSTM",
            "TCN"
        ]:

            for tsd in TSEEDS:

                model = load_seq(
                    fam,
                    tsd
                )

                prediction = pred_seq(
                    model,
                    Xss
                )

                assert len(
                    prediction
                ) == NTEST

                result = met(
                    y,
                    prediction
                )

                rows.append({
                    "condition": cond,
                    "perturbation_seed": psd,
                    "training_seed": tsd,
                    "model": fam,
                    **result
                })


        # ====================================================
        # 7. GCN TOPOLOGY ABLATION
        # ====================================================

        for topo in [
            "G_native",
            "G_operational",
            "G_shuffled"
        ]:

            for tsd in TSEEDS:

                model = load_gcn(
                    topo,
                    tsd
                )

                pm = pred_graph(
                    model,
                    Xgs,
                    Ah[topo]
                )

                assert pm.shape == (
                    NSNAP,
                    NODES
                )


                # Convert graph snapshot/node structure
                # into sequence node-level order.
                prediction = np.array(
                    [
                        pm[s, n]
                        for s, n in seqmap
                    ],
                    dtype=np.float32
                )

                assert len(
                    prediction
                ) == NTEST


                # Graph predictions are now in sequence order.
                result = met(
                    y,
                    prediction
                )


                rows.append({
                    "condition": cond,
                    "perturbation_seed": psd,
                    "training_seed": tsd,
                    "model": topo,
                    **result
                })


        print(
            f"✓ completed {cond} | "
            f"perturbation seed = {psd}"
        )


# ============================================================
# 8. CREATE RAW RESULTS TABLE
# ============================================================

raw = pd.DataFrame(
    rows
)

audit = pd.DataFrame(
    audits
)


# ============================================================
# 9. BASIC INTEGRITY CHECKS
# ============================================================

assert len(raw) > 0

assert (
    raw["n"] == NTEST
).all()


expected_conditions = {
    "S2-M",
    "S2-N",
    "S2-C"
}

assert set(
    raw["condition"].unique()
) == expected_conditions


expected_models = {
    "Persistence",
    "Ridge",
    "RandomForest",
    "XGBoost",
    "LSTM",
    "TCN",
    "G_native",
    "G_operational",
    "G_shuffled"
}

assert set(
    raw["model"].unique()
) == expected_models


# ============================================================
# 10. EXPECTED NUMBER OF RUNS
# ============================================================

# Per perturbation seed:
#
# 4 deterministic models
# +
# 2 sequential models × 3 training seeds = 6
# +
# 3 GCN topologies × 3 training seeds = 9
#
# Total = 19 results per perturbation seed
#
# 3 conditions × 5 perturbation seeds × 19 = 285

expected_rows = (
    len(conditions)
    * len(PSEEDS)
    * (
        4
        + 2 * len(TSEEDS)
        + 3 * len(TSEEDS)
    )
)

print(
    "\nExpected metric rows:",
    expected_rows
)

print(
    "Observed metric rows:",
    len(raw)
)

assert len(raw) == expected_rows


# ============================================================
# 11. MISSINGNESS AUDIT
# ============================================================

missing_audit = audit[
    audit["condition"].isin(
        [
            "S2-M",
            "S2-C"
        ]
    )
].copy()


print(
    "\nRealized missingness:"
)

print(
    missing_audit[
        [
            "condition",
            "perturbation_seed",
            "realized_missing_pct"
        ]
    ].to_string(
        index=False
    )
)


assert missing_audit[
    "realized_missing_pct"
].between(
    9.5,
    10.5
).all()


# ============================================================
# 12. SAVE RAW RESULTS
# ============================================================

raw.to_csv(
    OUT / "01_S2_metrics_all_runs.csv",
    index=False
)

audit.to_csv(
    OUT / "02_perturbation_audit.csv",
    index=False
)


# ============================================================
# 13. QUICK RESULTS PREVIEW
# ============================================================

quick_summary = (
    raw
    .groupby(
        [
            "condition",
            "model"
        ],
        as_index=False
    )
    .agg(
        RMSE_mean=(
            "RMSE",
            "mean"
        ),
        RMSE_sd=(
            "RMSE",
            "std"
        ),
        MAE_mean=(
            "MAE",
            "mean"
        ),
        R2_mean=(
            "R2",
            "mean"
        )
    )
)


print(
    "\n"
    + "=" * 70
)

print(
    "S2 QUICK SUMMARY"
)

print(
    "=" * 70
)

print(
    quick_summary.to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 70
)

print(
    "✓ S2-M, S2-N AND S2-C COMPLETED"
)

print(
    "✓ Total metric rows:",
    len(raw)
)

print(
    "✓ Results saved to:",
    OUT
)

print(
    "=" * 70
)

## 7. Aggregate results and calculate degradation relative to S1

In [ ]:
# ============================================================
# CELL 7 — FINAL S2 AGGREGATION AND S1 -> S2 DEGRADATION
# ============================================================

# ============================================================
# 1. AGGREGATE ALL S2 RUNS
# ============================================================

summary = (
    raw
    .groupby(
        ["condition", "model"],
        as_index=False
    )
    .agg(
        runs=("RMSE", "size"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_sd=("RMSE", "std"),

        MAE_mean=("MAE", "mean"),
        MAE_sd=("MAE", "std"),

        R2_mean=("R2", "mean"),
        R2_sd=("R2", "std"),

        Bias_mean=("Bias", "mean"),
        Bias_sd=("Bias", "std"),

        Error_SD_mean=("Error_SD", "mean"),
        Error_SD_sd=("Error_SD", "std")
    )
)


# ============================================================
# 2. BUILD EXACT S1 REFERENCES
# ============================================================

refs = []


# ------------------------------------------------------------
# A. Deterministic tabular models
# Predictions are in frozen TABULAR order -> use y_tab
# ------------------------------------------------------------

for model_name in [
    "Persistence",
    "Ridge",
    "RandomForest",
    "XGBoost"
]:

    result = met(
        y_tab,
        S1[model_name]
    )

    refs.append({
        "model": model_name,

        "S1_runs": 1,

        "S1_RMSE_mean":
            result["RMSE"],

        "S1_RMSE_sd":
            0.0,

        "S1_MAE_mean":
            result["MAE"],

        "S1_MAE_sd":
            0.0,

        "S1_R2_mean":
            result["R2"],

        "S1_R2_sd":
            0.0,

        "S1_Bias_mean":
            result["Bias"],

        "S1_Error_SD_mean":
            result["Error_SD"]
    })


# ------------------------------------------------------------
# B. Sequential models
# Predictions are in SEQUENCE order -> use y
# ------------------------------------------------------------

for model_name in [
    "LSTM",
    "TCN"
]:

    model_results = []

    for i, seed in enumerate(TSEEDS):

        result = met(
            y,
            S1[model_name][i]
        )

        model_results.append(
            result
        )


    refs.append({
        "model": model_name,

        "S1_runs":
            len(model_results),

        "S1_RMSE_mean":
            np.mean([
                r["RMSE"]
                for r in model_results
            ]),

        "S1_RMSE_sd":
            np.std(
                [
                    r["RMSE"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_MAE_mean":
            np.mean([
                r["MAE"]
                for r in model_results
            ]),

        "S1_MAE_sd":
            np.std(
                [
                    r["MAE"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_R2_mean":
            np.mean([
                r["R2"]
                for r in model_results
            ]),

        "S1_R2_sd":
            np.std(
                [
                    r["R2"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_Bias_mean":
            np.mean([
                r["Bias"]
                for r in model_results
            ]),

        "S1_Error_SD_mean":
            np.mean([
                r["Error_SD"]
                for r in model_results
            ])
    })


# ------------------------------------------------------------
# C. GCN topology variants
# Predictions converted to SEQUENCE order -> use y
# ------------------------------------------------------------

for model_name in [
    "G_native",
    "G_operational",
    "G_shuffled"
]:

    model_results = []

    for i, seed in enumerate(TSEEDS):

        result = met(
            y,
            S1[model_name][i]
        )

        model_results.append(
            result
        )


    refs.append({
        "model": model_name,

        "S1_runs":
            len(model_results),

        "S1_RMSE_mean":
            np.mean([
                r["RMSE"]
                for r in model_results
            ]),

        "S1_RMSE_sd":
            np.std(
                [
                    r["RMSE"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_MAE_mean":
            np.mean([
                r["MAE"]
                for r in model_results
            ]),

        "S1_MAE_sd":
            np.std(
                [
                    r["MAE"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_R2_mean":
            np.mean([
                r["R2"]
                for r in model_results
            ]),

        "S1_R2_sd":
            np.std(
                [
                    r["R2"]
                    for r in model_results
                ],
                ddof=1
            ),

        "S1_Bias_mean":
            np.mean([
                r["Bias"]
                for r in model_results
            ]),

        "S1_Error_SD_mean":
            np.mean([
                r["Error_SD"]
                for r in model_results
            ])
    })


refs = pd.DataFrame(refs)


# ============================================================
# 3. MERGE S1 REFERENCES WITH S2
# ============================================================

summary = summary.merge(
    refs,
    on="model",
    how="left",
    validate="many_to_one"
)


assert summary[
    "S1_RMSE_mean"
].notna().all()


# ============================================================
# 4. CALCULATE ABSOLUTE AND RELATIVE DEGRADATION
# ============================================================

summary[
    "Delta_RMSE"
] = (
    summary["RMSE_mean"]
    - summary["S1_RMSE_mean"]
)


summary[
    "Delta_RMSE_pct"
] = (
    100
    * summary["Delta_RMSE"]
    / summary["S1_RMSE_mean"]
)


summary[
    "Delta_MAE"
] = (
    summary["MAE_mean"]
    - summary["S1_MAE_mean"]
)


summary[
    "Delta_MAE_pct"
] = (
    100
    * summary["Delta_MAE"]
    / summary["S1_MAE_mean"]
)


summary[
    "Delta_R2"
] = (
    summary["R2_mean"]
    - summary["S1_R2_mean"]
)


# ============================================================
# 5. SAVE COMPLETE S2 SUMMARY
# ============================================================

summary.to_csv(
    OUT / "03_S2_summary_mean_sd.csv",
    index=False
)


# ============================================================
# 6. PRIMARY MANUSCRIPT TABLE
# ============================================================

primary_models = [
    "Persistence",
    "Ridge",
    "RandomForest",
    "XGBoost",
    "LSTM",
    "TCN",
    "G_native"
]


primary = summary[
    summary["model"].isin(
        primary_models
    )
].copy()


primary["model"] = pd.Categorical(
    primary["model"],
    categories=primary_models,
    ordered=True
)


condition_order = [
    "S2-M",
    "S2-N",
    "S2-C"
]


primary["condition"] = pd.Categorical(
    primary["condition"],
    categories=condition_order,
    ordered=True
)


primary = primary.sort_values(
    [
        "condition",
        "model"
    ]
)


primary.to_csv(
    OUT / "04_S2_primary_manuscript_table.csv",
    index=False
)


# ============================================================
# 7. GRAPH TOPOLOGY SENSITIVITY TABLE
# ============================================================

graph_models = [
    "G_native",
    "G_operational",
    "G_shuffled"
]


graph_sensitivity = summary[
    summary["model"].isin(
        graph_models
    )
].copy()


graph_sensitivity[
    "condition"
] = pd.Categorical(
    graph_sensitivity["condition"],
    categories=condition_order,
    ordered=True
)


graph_sensitivity[
    "model"
] = pd.Categorical(
    graph_sensitivity["model"],
    categories=graph_models,
    ordered=True
)


graph_sensitivity = (
    graph_sensitivity
    .sort_values(
        [
            "condition",
            "model"
        ]
    )
)


graph_sensitivity.to_csv(
    OUT /
    "05_S2_graph_topology_sensitivity.csv",
    index=False
)


# ============================================================
# 8. S1 REFERENCE TABLE
# ============================================================

refs.to_csv(
    OUT / "06_S1_reference_metrics_for_S2.csv",
    index=False
)


# ============================================================
# 9. FINAL INTEGRITY CHECKS
# ============================================================

assert len(
    summary
) == (
    3 * 9
)

assert set(
    summary["condition"].astype(str)
) == {
    "S2-M",
    "S2-N",
    "S2-C"
}


# S2 is a perturbation robustness experiment.
# We retain negative Delta values if they occur;
# we do NOT force degradation to be positive.
assert np.isfinite(
    summary[
        [
            "RMSE_mean",
            "MAE_mean",
            "R2_mean",
            "Delta_RMSE",
            "Delta_RMSE_pct"
        ]
    ].to_numpy()
).all()


# ============================================================
# 10. DISPLAY PRIMARY RESULTS
# ============================================================

display_columns = [
    "condition",
    "model",
    "runs",

    "S1_RMSE_mean",
    "S1_RMSE_sd",

    "RMSE_mean",
    "RMSE_sd",

    "Delta_RMSE",
    "Delta_RMSE_pct",

    "MAE_mean",
    "R2_mean"
]


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL S2 PRIMARY ROBUSTNESS RESULTS"
)

print(
    "=" * 100
)


display(
    primary[
        display_columns
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "GCN TOPOLOGY SENSITIVITY UNDER S2"
)

print(
    "=" * 100
)


display(
    graph_sensitivity[
        [
            "condition",
            "model",
            "S1_RMSE_mean",
            "RMSE_mean",
            "RMSE_sd",
            "Delta_RMSE",
            "Delta_RMSE_pct"
        ]
    ]
)


print(
    "\n✓ Final S2 aggregation completed"
)

print(
    "✓ S1 references computed with the correct target ordering"
)

print(
    "✓ S2 degradation calculated relative to frozen S1"
)

print(
    "✓ Results saved to:",
    OUT
)

## 8. Integrity checks and reproducibility manifest

In [ ]:
# ============================================================
# CELL 8 — INTEGRITY CHECKS AND REPRODUCIBILITY MANIFEST
# ============================================================

# ------------------------------------------------------------
# 1. BASIC SAMPLE-SIZE CHECK
# ------------------------------------------------------------

assert (raw["n"] == NTEST).all(), \
    "Unexpected number of TEST observations in S2 metrics."


# ------------------------------------------------------------
# 2. EXPECTED NUMBER OF METRIC ROWS
# ------------------------------------------------------------

expected_metric_rows = (
    3
    * len(PSEEDS)
    * (
        4
        + 2 * len(TSEEDS)
        + 3 * len(TSEEDS)
    )
)

assert len(raw) == expected_metric_rows, \
    f"Expected {expected_metric_rows} metric rows, got {len(raw)}."


# ------------------------------------------------------------
# 3. CONDITION CHECK
# ------------------------------------------------------------

assert set(raw["condition"].unique()) == {
    "S2-M",
    "S2-N",
    "S2-C"
}


# ------------------------------------------------------------
# 4. PERTURBATION-SEED CHECK
# ------------------------------------------------------------

for condition in [
    "S2-M",
    "S2-N",
    "S2-C"
]:
    
    observed_seeds = set(
        raw.loc[
            raw["condition"] == condition,
            "perturbation_seed"
        ].unique()
    )
    
    assert observed_seeds == set(PSEEDS), \
        f"Perturbation seeds mismatch for {condition}"


# ------------------------------------------------------------
# 5. MISSINGNESS AUDIT
# ------------------------------------------------------------

q = audit[
    audit["condition"].isin(
        [
            "S2-M",
            "S2-C"
        ]
    )
].copy()


assert len(q) == (
    2 * len(PSEEDS)
)


assert q[
    "realized_missing_pct"
].between(
    9.5,
    10.5
).all(), \
    "Realized MCAR missingness outside expected tolerance."


# ------------------------------------------------------------
# 6. CHECK S2-N HAS NO ARTIFICIAL MISSINGNESS
# ------------------------------------------------------------

q_noise = audit[
    audit["condition"] == "S2-N"
]

assert (
    q_noise["masked_values"] == 0
).all(), \
    "Unexpected missing values generated in S2-N."


# ------------------------------------------------------------
# 7. CHECK TARGETS / METRICS
# ------------------------------------------------------------

assert np.isfinite(
    raw[
        [
            "RMSE",
            "MAE",
            "R2",
            "Bias",
            "Error_SD"
        ]
    ].to_numpy()
).all(), \
    "Non-finite metric detected."


# ------------------------------------------------------------
# 8. REPRODUCIBILITY MANIFEST
# ------------------------------------------------------------

manifest = {

    "scenario":
        "S2 controlled perturbation robustness",

    "test_samples":
        int(NTEST),

    "test_graph_snapshots":
        int(NSNAP),

    "conditions": [
        "S2-M",
        "S2-N",
        "S2-C"
    ],

    "missingness": {
        "mechanism":
            "MCAR",

        "unit":
            "individual input measurement",

        "rate":
            float(MISSING_RATE)
    },

    "noise": {
        "distribution":
            "Gaussian zero-mean",

        "definition":
            "0.05 * TRAIN variable scale",

        "fraction":
            float(NOISE_FRACTION),

        "sigmas": dict(
            zip(
                VARIABLES,
                [
                    float(x)
                    for x in noise_sigma
                ]
            )
        )
    },

    "imputation":
        (
            "causal forward fill within each 5-step "
            "input history; TRAIN variable mean "
            "fallback for leading missing values"
        ),

    "perturbation_application":
        (
            "perturbation generated once on the common "
            "unscaled graph physical tensor before "
            "representation-specific mapping"
        ),

    "perturbation_seeds":
        [
            int(x)
            for x in PSEEDS
        ],

    "frozen_training_seeds":
        [
            int(x)
            for x in TSEEDS
        ],

    "targets_perturbed":
        False,

    "training_data_perturbed":
        False,

    "models_retrained_for_S2":
        False,

    "hyperparameter_retuning":
        False,

    "primary_gcn":
        "G_native",

    "graph_sensitivity": [
        "G_native",
        "G_operational",
        "G_shuffled"
    ],

    "tabular_alignment":
        (
            "exact UUID + target_timestamp mapping "
            "to frozen tabular TEST row order"
        ),

    "sequential_alignment":
        "frozen sequence TEST order",

    "graph_to_sequence_alignment":
        "frozen snapshot-node seqmap",

    "expected_metric_rows":
        int(expected_metric_rows)
}


# ------------------------------------------------------------
# 9. SAVE MANIFEST
# ------------------------------------------------------------

manifest_path = (
    OUT /
    "07_S2_reproducibility_manifest.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("=" * 70)

print(
    "✓ S2 integrity checks passed"
)

print(
    "✓ TEST samples:",
    NTEST
)

print(
    "✓ Metric rows:",
    len(raw)
)

print(
    "✓ Perturbation seeds:",
    PSEEDS
)

print(
    "✓ Frozen training seeds:",
    TSEEDS
)

print(
    "✓ Targets perturbed: False"
)

print(
    "✓ Training data perturbed: False"
)

print(
    "✓ Models retrained for S2: False"
)

print(
    "✓ Hyperparameter retuning: False"
)

print(
    "✓ Manifest:",
    manifest_path
)

print(
    "✓ Outputs:",
    OUT
)

print("=" * 70)